# 01. Чтение и подготовка данных

**Цель ноутбука** — из сырых логов Google Analytics собрать одну чистую таблицу, где **одна строка = один визит**:
признаки визита + таргет (было ли целевое действие) + несколько поведенческих показателей для анализа.

**Данные:**
- `ga_sessions.pkl` — визиты: дата и время, UTM-метки, устройство, гео, номер визита клиента;
- `ga_hits.pkl` — события внутри визитов: страницы, клики, действия (`event_action`).

**Результат:** `data/interim/sessions_clean.pkl` — вход для EDA (`02_eda.ipynb`).
Модель (`03_model.ipynb`) строит признаки тем же модулем `src/features.py`, поэтому таргет и признаки во всех ноутбуках совпадают.

In [1]:
import gc, os, sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

root = Path(os.getcwd()).parent
sys.path.insert(0, str(root))
from src.features import TARGET_ACTIONS, ORGANIC_MEDIUMS, build_features, session_targets

raw = root / 'data' / 'raw'
interim = root / 'data' / 'interim'
interim.mkdir(parents=True, exist_ok=True)

## 1. Визиты (`ga_sessions`)

In [2]:
sessions = pd.read_pickle(raw / 'ga_sessions.pkl')
print(f'Визитов: {len(sessions):,}, колонок: {sessions.shape[1]}')
print(f'Уникальных session_id: {sessions["session_id"].nunique():,}, клиентов: {sessions["client_id"].nunique():,}')
sessions.head(3)

Визитов: 1,860,042, колонок: 18
Уникальных session_id: 1,860,042, клиентов: 1,391,719


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_model,device_screen_resolution,device_browser,geo_country,geo_city
0,9055434745589932991.1637753792.1637753792,2108382700.1637753791,2021-11-24,14:36:32,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Zlatoust
1,905544597018549464.1636867290.1636867290,210838531.1636867288,2021-11-14,08:21:30,1,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,IGUCNvHlhfHpROGclCit,mobile,Android,Samsung,NaN,385x854,Samsung Internet,Russia,Moscow
2,9055446045651783499.1640648526.1640648526,2108385331.1640648523,2021-12-28,02:42:06,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Krasnoyarsk


### Пропуски и «пустые» значения
В GA пропуск бывает не только `NaN`, но и строкой `(not set)`. Считаем оба варианта.
`(none)` в `utm_medium` — **не пропуск**: по глоссарию задачи это прямой заход, он относится к органическому трафику.

In [3]:
def missing_report(df):
    rep = pd.DataFrame({
        'NaN, %': df.isna().mean() * 100,
        '(not set), %': df.apply(lambda s: (s.astype(str) == '(not set)').mean() * 100 if s.dtype == object else 0.0),
    })
    rep['всего, %'] = rep.sum(axis=1)
    return rep[rep['всего, %'] > 0].sort_values('всего, %', ascending=False)

missing_report(sessions)

,"NaN, %","(not set), %","всего, %"
device_model,99.122,0.000,99.122
utm_keyword,58.174,0.000,58.174
device_os,57.533,0.020,57.553
utm_adcontent,18.043,0.000,18.043
utm_campaign,11.806,0.000,11.806
device_brand,6.380,0.943,7.324
geo_city,0.000,4.203,4.203
geo_country,0.000,0.062,0.062
utm_medium,0.000,0.026,0.026
utm_source,0.005,0.000,0.005


**Решения по пропускам:**
- `device_model` — пусто почти всегда (~99%), колонку не используем;
- остальные категориальные пропуски (`utm_keyword`, `device_os`, `utm_adcontent`, `utm_campaign`, `device_brand`…)
  заполняем отдельной категорией `(missing)`: сам факт пропуска информативен
  (например, у органики нет кампании и ключевого слова), удалять такие визиты нельзя — их слишком много;
- разрешение экрана разбираем на ширину и высоту; `(not set)` становится пропуском — CatBoost и импьютер в модели
  умеют с ними работать.

In [4]:
print('Типы утм-трафика (utm_medium), топ-12:')
display(sessions['utm_medium'].value_counts().head(12).to_frame('визитов'))
print('Полных дубликатов строк:', sessions.duplicated().sum())

Типы утм-трафика (utm_medium), топ-12:


,визитов
utm_medium,
banner,552272
cpc,434794
(none),300575
cpm,242083
referral,152050
organic,63034
email,29240
push,28035
stories,10582


Полных дубликатов строк: 0


## 2. События (`ga_hits`) и таргет

In [5]:
hits = pd.read_pickle(raw / 'ga_hits.pkl')[['session_id', 'hit_number', 'hit_time', 'hit_page_path', 'event_action']]
print(f'Событий: {len(hits):,}, визитов с событиями: {hits["session_id"].nunique():,}')
print('Пропуски, %:'); display((hits.isna().mean() * 100).round(2).to_frame('NaN, %'))

Событий: 15,726,470, визитов с событиями: 1,734,610
Пропуски, %:


,"NaN, %"
session_id,0.000
hit_number,0.000
hit_time,58.250
hit_page_path,0.000
event_action,0.000


### Что считаем целевым действием
По брифу целевое действие — «Оставить заявку», «Заказать звонок», «Открыть диалог» и т.п.
В логах это события `event_action` с префиксом `sub_`, которые означают **контакт с компанией**:

In [6]:
sub = hits.loc[hits['event_action'].astype(str).str.startswith('sub_'), 'event_action'].value_counts().to_frame('событий')
sub['целевое'] = sub.index.isin(TARGET_ACTIONS)
sub

,событий,целевое
event_action,,
sub_landing,1798117,False
sub_view_cars_click,791515,False
sub_car_page,130310,False
sub_header_link_click,79192,False
sub_offer_click,69291,False
sub_faq,56034,False
sub_view_faq_click,43232,False
sub_car_claim_click,37928,True
sub_open_dialog_click,25870,True


Целевые (8 штук): клик и отправка заявки на авто, открытие диалога, вопрос, звонок, обратный звонок,
успешная отправка формы, запрос по авто.
**Не считаем целевыми** просмотр лендинга, каталога и карточки авто (`sub_landing`, `sub_view_cars_click`, `sub_car_page`),
клики по меню и FAQ — это интерес, а не обращение.

**Таргет визита** = 1, если в визите было хотя бы одно целевое действие.

In [7]:
target = session_targets(hits)

# Поведенческие показатели визита — ТОЛЬКО для EDA (известны лишь после визита, в модель не идут)
behaviour = hits.groupby('session_id').agg(
    n_hits=('hit_number', 'max'),
    n_pages=('hit_page_path', 'nunique'),
    duration_sec=('hit_time', 'max'),
)
behaviour['duration_sec'] = behaviour['duration_sec'] / 1000
del hits; gc.collect()
print(f'Визитов с целевым действием: {target.sum():,} из {len(target):,} ({target.mean():.2%})')

Визитов с целевым действием: 50,760 из 1,734,610 (2.93%)


## 3. Сборка таблицы визитов

Берём только визиты, по которым есть события (иначе таргет неизвестен), и добавляем:
- **признаки для модели** — из `src/features.py` (то же, что увидит API);
- **дату визита** — для анализа динамики;
- **поведенческие показатели** — для EDA.

In [8]:
n_before = len(sessions)
sessions = sessions[sessions['session_id'].isin(target.index)].reset_index(drop=True)
print(f'Визитов без событий (исключены): {n_before - len(sessions):,} ({1 - len(sessions) / n_before:.1%})')

visit_dt = pd.to_datetime(sessions['visit_date'], errors='coerce').dt.normalize() + \
           pd.to_timedelta(sessions['visit_time'].astype(str), errors='coerce')

df = build_features(sessions)
df.insert(0, 'session_id', sessions['session_id'].values)
df.insert(1, 'client_id', sessions['client_id'].values)
df.insert(2, 'visit_dt', visit_dt.values)
df = df.join(behaviour, on='session_id')
df['target'] = df['session_id'].map(target).astype('int8')
del sessions; gc.collect()

print(f'Итог: {len(df):,} визитов × {df.shape[1]} колонок, конверсия {df["target"].mean():.2%}')
print('Дубликатов session_id:', df['session_id'].duplicated().sum())
print('Пропуски после обработки, %:')
display((df.isna().mean() * 100).round(2).loc[lambda s: s > 0].to_frame('NaN, %'))
df.head()

Визитов без событий (исключены): 127,776 (6.9%)
Итог: 1,732,266 визитов × 29 колонок, конверсия 2.90%
Дубликатов session_id: 0
Пропуски после обработки, %:


,"NaN, %"
duration_sec,57.660


,session_id,client_id,visit_dt,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_browser,geo_country,geo_city,source_medium,device_category_os,visit_number,is_new_visitor,visit_hour,visit_weekday,is_weekend,is_organic,screen_w,screen_h,screen_ratio,n_hits,n_pages,duration_sec,target
0,9055434745589932991.1637753792.1637753792,2108382700.1637753791,2021-11-24 14:36:32,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,Chrome,Russia,Zlatoust,ZpYIoDJMcFzVoPFsHGJL | banner,mobile | Android,1,1,14,2,0,0,360.000,720.000,2.000,4,1,46.592,0
1,905544597018549464.1636867290.1636867290,210838531.1636867288,2021-11-14 08:21:30,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,IGUCNvHlhfHpROGclCit,mobile,Android,Samsung,Samsung Internet,Russia,Moscow,MvfHsxITijuriZxsqZqt | cpm,mobile | Android,1,1,8,6,1,0,385.000,854.000,2.218,3,1,0.921,0
2,9055446045651783499.1640648526.1640648526,2108385331.1640648523,2021-12-28 02:42:06,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,Chrome,Russia,Krasnoyarsk,ZpYIoDJMcFzVoPFsHGJL | banner,mobile | Android,1,1,2,1,0,0,360.000,720.000,2.000,20,2,116.142,0
3,9055447046360770272.1622255328.1622255328,2108385564.1622255328,2021-05-29 05:00:00,kjsLglQLzykiRbcDiGcD,cpc,(missing),NOBKLgtuvqYWkXQHeYWM,(missing),mobile,(missing),Xiaomi,Chrome,Russia,Moscow,kjsLglQLzykiRbcDiGcD | cpc,mobile | (missing),1,1,5,5,1,0,393.000,786.000,2.000,7,1,NaN,0
4,9055447046360770272.1622255345.1622255345,2108385564.1622255328,2021-05-29 05:00:00,kjsLglQLzykiRbcDiGcD,cpc,(missing),(missing),(missing),mobile,(missing),Xiaomi,Chrome,Russia,Moscow,kjsLglQLzykiRbcDiGcD | cpc,mobile | (missing),2,0,5,5,1,0,393.000,786.000,2.000,3,1,NaN,0


## 4. Сохранение

In [9]:
df.to_pickle(interim / 'sessions_clean.pkl')
print('Сохранено:', (interim / 'sessions_clean.pkl').relative_to(root), f'({len(df):,} строк)')

Сохранено: data/interim/sessions_clean.pkl (1,732,266 строк)


## Итог
- Визитов с событиями — ~1,73 млн, конверсия в целевое действие ≈ 2,9% (сильный дисбаланс классов).
- Пропуски категориальных полей заполнены категорией `(missing)`, `(none)` в `utm_medium` сохранён как органика.
- Таргет и признаки построены общим модулем `src/features.py` — те же, что в модели и API.